In [ ]:
import csv
import os
import cobra
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp as solver
from cobra import Model, Reaction, Metabolite
from cobra.flux_analysis import pfba
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib_venn import venn2
from scipy.integrate import solve_ivp
from scipy.optimize import minimize
from matplotlib.colors import ListedColormap
import string
import sklearn
from sklearn.metrics import confusion_matrix, matthews_corrcoef
import matplotlib.patches as mpatches

In [ ]:
# Load YCFA in-silico medium
YCFA_file = '/Users/lishijia/Documents/simulations/BacArena/nutrints_YCFA.csv'
df = pd.read_csv(YCFA_file,header=0)
YCFA_subs = df.iloc[:,0]
YCFA_con = df.iloc[:,2]

# Get substrate list in YCFA
sub_list = []
for i in range(len(YCFA_subs)):
    sub_list.append(YCFA_subs[i])
    
# Get substrate concentrations in YCFA and store it in a list object
Initial_state = []
for i in range(len(YCFA_con)):
    Initial_state.append(YCFA_con[i])
Initial_state = np.array(Initial_state)

In [ ]:
def Trans_OD_biomass(exp_df,carbon,k_factor):
    exp_df['Time'] = pd.to_numeric(exp_df['Time'],errors='coerce')
    exp_df['MeanOD'] = pd.to_numeric(exp_df['MeanOD'],errors='coerce')
    exp_df['StdOD'] = pd.to_numeric(exp_df['StdOD'],errors='coerce')
    exp_df = exp_df.dropna()

    exp_data_C = exp_df[exp_df['CarbonSource'] == carbon].sort_values('Time')

    # Transform OD to Biomass
    exp_data_C['Biomass_Exp'] = exp_data_C['MeanOD']*k_factor
    exp_data_C['Biomass_Std_Scaled'] = exp_data_C['StdOD']*k_factor

    return(exp_data_C)

def YCFA_medium_def(model, products, EXP_data, Carbon='GMC', C_con=0):
    YCFA_subs_use = [reac for reac in YCFA_subs if reac in model.exchanges]
    YCFA_new = {k: v for k, v in zip(sub_list, Initial_state) if k in YCFA_subs_use}

    if Carbon != 'GMC':
        default_carbons = ['EX_glc_D(e)', 'EX_malt(e)', 'EX_cellb(e)']
        for r in default_carbons:
            YCFA_new.pop(r, None)
        if Carbon in model.exchanges:
            YCFA_new[Carbon] = C_con
        else:
            print(f"Warning: cannot find {Carbon} in model")

    for p in products:
        if p in model.exchanges:
            if p not in YCFA_new:
                YCFA_new[p] = 0.0
        else:
            print(f"Pass {p}: it doesn't present in exchanges list")

    YCFA_sub_new = list(YCFA_new.keys())
    YCFA_con_new = np.array(list(YCFA_new.values()))
    
    initial_biomass = EXP_data.Biomass_Exp.values[0]
    YCFA_con_new = np.append(YCFA_con_new, initial_biomass)

    return YCFA_sub_new, YCFA_con_new

def limit_subs_dect(model, Sub, Con, tol=1e-6):
    with model:
        for i in range(len(Sub)):
            if Sub[i] in model.exchanges:
                model.exchanges.get_by_id(Sub[i]).lower_bound = -Con[i]
            else:
                continue

        solution = model.optimize()

        if solution.status != 'optimal':
            raise RuntimeError(f"Model optimization failed: {solution.status}")

        reduced_costs = solution.reduced_costs

        print('--- Analysis of Substrate Exchange Reactions Limiting Model Growth ---')
        limiting_exchanges = []

        for rxn in model.exchanges:
            flux = solution.fluxes[rxn.id]
            low_bound = rxn.lower_bound
            rc = reduced_costs[rxn.id]
            
            if (
                flux < -tol and
                abs(flux - low_bound) < tol and
                abs(rc) > tol
            ):
                limiting_exchanges.append(
                    (rxn.id, rxn.name, flux, rc)
                )

        limiting_exchanges.sort(
            key=lambda x: abs(x[3]), reverse=True
        )

        df_limiting = pd.DataFrame(
            limiting_exchanges,
            columns=["Reaction ID", "Name", "Current Flux", "Reduced Cost"]
        )
        return df_limiting

def plot_parameter_sensitivity_heatmap(model, exp_data, lag_end_time, carbon_ids, 
                                       media_sub_ids, initial_concs, substrate_limit_val,
                                       vmax_range=(1, 20, 15), km_range=(0.01, 3.0, 15),
                                       scfa_ids=None):

    if scfa_ids is None:
        scfa_ids = {
            'Acetate': 'EX_ac(e)',
            'Butyrate': 'EX_but(e)',
            'Lactate': 'EX_lac_L(e)',  
            'Formate': 'EX_for(e)',
            'Propionate': 'EX_ppa(e)',
            'Succinate': 'EX_succ(e)'
        }

    valid_media_ids = [rid for rid in media_sub_ids if rid in model.exchanges]
    valid_carbon_ids = [rid for rid in carbon_ids if rid in model.exchanges]
    if not valid_carbon_ids:
        raise ValueError(f"错误：核心碳源 {carbon_ids} 不在模型中！")
    
    s_idx = {rid: i for i, rid in enumerate(valid_media_ids)}
    
    od_col = 'OD_Exp' if 'OD_Exp' in exp_data.columns else 'MeanOD'
    std_col = 'OD_Std' if 'OD_Std' in exp_data.columns else 'StdOD'
    
    max_idx = exp_data[od_col].idxmax()
    mask = (exp_data['Time'] >= lag_end_time) & (exp_data.index <= max_idx)
    df_log = exp_data.loc[mask].copy()
    t_log = df_log['Time'].values
    od_exp_vals = df_log[od_col].values
    std_exp_vals = df_log[std_col].values
    
    norm_exp = od_exp_vals / np.max(od_exp_vals)
    tss = np.sum((norm_exp - np.mean(norm_exp))**2)

    valid_rxn_objs = {rid: model.reactions.get_by_id(rid) for rid in valid_media_ids}
    
    original_bounds = {rid: valid_rxn_objs[rid].lower_bound for rid in valid_media_ids}

    def mu_step(conc_vec, v_dict, k_dict):
        conc_vec = np.maximum(conc_vec, 0)
        if all(conc_vec[s_idx[rid]] < 1e-6 for rid in valid_carbon_ids):
            return [0.0] * (len(valid_media_ids) + 1)
        
        for rid in valid_media_ids:
            s_c = conc_vec[s_idx[rid]]
            if rid in v_dict:
                v_lim = (v_dict[rid] * s_c) / (k_dict[rid] + s_c)
                valid_rxn_objs[rid].lower_bound = -v_lim
            else:
                valid_rxn_objs[rid].lower_bound = -s_c
                
        model.optimize()
        if model.solver.status == 'optimal':
            res = [valid_rxn_objs[rid].flux for rid in valid_media_ids]
            res.append(model.objective.value)
            return res
        return [0.0] * (len(valid_media_ids) + 1)

    def derivs(t, y, v_d, k_d):
        mu_res = mu_step(y, v_d, k_d)
        d_y = np.zeros(len(y))
        d_y[:-1] = np.array(mu_res[:-1]) * y[-1]
        d_y[-1] = mu_res[-1] * y[-1]
        return d_y

    v_axis = np.linspace(vmax_range[0], vmax_range[1], vmax_range[2])
    k_axis = np.linspace(km_range[0], km_range[1], km_range[2])
    r2_matrix = np.full((len(k_axis), len(v_axis)), -1.0) 

    print(f"Scanning {len(v_axis)}x{len(k_axis)} parameter sets...")
    for i, k_val in enumerate(k_axis):
        for j, v_val in enumerate(v_axis):
            v_p, k_p = {valid_carbon_ids[0]: v_val}, {valid_carbon_ids[0]: k_val}
            y0 = np.append(initial_concs[:len(valid_media_ids)], od_exp_vals[0])
            for rid in valid_carbon_ids: y0[s_idx[rid]] = substrate_limit_val
            
            sol = solve_ivp(derivs, (t_log[0], t_log[-1]), y0, args=(v_p, k_p), 
                            t_eval=t_log, method='RK23', rtol=1e-3, atol=1e-6)
            
            if sol.success:
                sim_n = sol.y[-1] / (np.max(sol.y[-1]) + 1e-9)
                r2_matrix[i, j] = 1 - (np.sum((sim_n - norm_exp)**2) / tss)

    idx_g = np.unravel_index(np.argmax(r2_matrix), r2_matrix.shape)
    best_v, best_k = v_axis[idx_g[1]], k_axis[idx_g[0]]
    max_r2 = np.max(r2_matrix)

    df_heatmap = pd.DataFrame(r2_matrix, index=np.round(k_axis, 3), columns=np.round(v_axis, 2))
    df_heatmap.index.name = 'Km'
    df_heatmap.columns.name = 'Vmax'

    v_p_best, k_p_best = {valid_carbon_ids[0]: best_v}, {valid_carbon_ids[0]: best_k}
    y0_best = np.append(initial_concs[:len(valid_media_ids)], od_exp_vals[0])
    for rid in valid_carbon_ids: y0_best[s_idx[rid]] = substrate_limit_val
    
    sol_best = solve_ivp(derivs, (t_log[0], t_log[-1]), y0_best, args=(v_p_best, k_p_best), 
                         t_eval=t_log, method='RK23', rtol=1e-3, atol=1e-6)

    df_time_series = pd.DataFrame({
        'Time': t_log,
        'Exp_OD': od_exp_vals,
        'Exp_OD_Std': std_exp_vals,
        'Sim_Biomass': sol_best.y[-1]
    })

    for scfa_name, exch_id in scfa_ids.items():
        if exch_id in s_idx:
            metabolite_idx = s_idx[exch_id]
            initial_background_conc = y0_best[metabolite_idx]
            net_production = sol_best.y[metabolite_idx] - initial_background_conc
            net_production = np.where(np.abs(net_production) < 1e-9, 0.0, net_production)
            df_time_series[f'Sim_{scfa_name}'] = net_production
        else:
            df_time_series[f'Sim_{scfa_name}'] = 0.0

    print(f"Best paras：Vmax={best_v:.2f}, Km={best_k:.3f}, R2={max_r2:.4f}")
    
    for rid, orig_bound in original_bounds.items():
        valid_rxn_objs[rid].lower_bound = orig_bound
    
    return {
        "metadata": {
            "carbon_id": valid_carbon_ids[0],
            "best_vmax": best_v, "best_km": best_k, "max_r2": max_r2
        },
        "heatmap_data": df_heatmap,
        "time_series_data": df_time_series
    }

def run_local_sensitivity_analysis(model, exp_data, lag_end_time, carbon_ids, media_sub_ids, 
                                   initial_concs, substrate_limit_val, 
                                   best_vmax, best_km, grid_size=20, plot_individual=False):
    if isinstance(carbon_ids, str):
        carbon_ids = [carbon_ids]
        
    valid_carbon_ids = [rid for rid in carbon_ids if rid in model.exchanges]
    primary_carbon = valid_carbon_ids[0] # 动力学参数只作用于第一个核心碳源

    print(f"Sensitive analysis (carbon: {primary_carbon}, para basements: Vmax={best_vmax:.2f}, Km={best_km:.3f})...")
    
    valid_media_ids = [rid for rid in media_sub_ids if rid in model.exchanges]
    s_idx = {rid: i for i, rid in enumerate(valid_media_ids)}
    
    od_col = 'OD_Exp' if 'OD_Exp' in exp_data.columns else 'MeanOD'
    max_idx = exp_data[od_col].idxmax()
    mask = (exp_data['Time'] >= lag_end_time) & (exp_data.index <= max_idx)
    df_log = exp_data.loc[mask].copy()
    
    if df_log.empty:
        raise ValueError("Error: Lag_end_time greater than OD time")

    t_log = df_log['Time'].values
    norm_exp = df_log[od_col].values / np.max(df_log[od_col].values)
    tss = np.sum((norm_exp - np.mean(norm_exp))**2)

    valid_rxn_objs = {rid: model.reactions.get_by_id(rid) for rid in valid_media_ids}
    original_bounds = {rid: valid_rxn_objs[rid].lower_bound for rid in valid_media_ids}

    def evaluate_r2(v_val, k_val):
        def mu_step(conc_vec):
            conc_vec = np.maximum(conc_vec, 0)
            
            if all(conc_vec[s_idx[rid]] < 1e-6 for rid in valid_carbon_ids): 
                return [0.0] * (len(valid_media_ids) + 1)
                
            for rid in valid_media_ids:
                s_c = conc_vec[s_idx[rid]]
                if rid == primary_carbon: 
                    valid_rxn_objs[rid].lower_bound = - (v_val * s_c) / (k_val + s_c)
                else: 
                    valid_rxn_objs[rid].lower_bound = -s_c
            model.optimize()
            if model.solver.status == 'optimal':
                res = [valid_rxn_objs[rid].flux for rid in valid_media_ids]
                res.append(model.objective.value)
                return res
            return [0.0] * (len(valid_media_ids) + 1)

        def derivs(t, y):
            mu_res = mu_step(y)
            d_y = np.zeros(len(y))
            d_y[:-1] = np.array(mu_res[:-1]) * y[-1]
            d_y[-1] = mu_res[-1] * y[-1]
            return d_y

        y0 = np.append(initial_concs[:len(valid_media_ids)], df_log[od_col].values[0])
        
        for rid in valid_carbon_ids:
            y0[s_idx[rid]] = substrate_limit_val
            
        sol = solve_ivp(derivs, (t_log[0], t_log[-1]), y0, t_eval=t_log, method='RK23', rtol=1e-3, atol=1e-6)
        
        if sol.success:
            sim_n = sol.y[-1] / (np.max(sol.y[-1]) + 1e-9)
            return 1 - (np.sum((sim_n - norm_exp)**2) / tss)
        return -1.0

    v_axis = np.linspace(best_vmax * 0.5, best_vmax * 1.5, grid_size)
    k_axis = np.linspace(best_km * 0.5, best_km * 1.5, grid_size)
    V_grid, K_grid = np.meshgrid(v_axis, k_axis)
    R2_matrix = np.zeros_like(V_grid)

    for i in range(grid_size):
        for j in range(grid_size):
            R2_matrix[i, j] = evaluate_r2(V_grid[i, j], K_grid[i, j])

    for rid, orig_bound in original_bounds.items():
        valid_rxn_objs[rid].lower_bound = orig_bound

    base_r2 = evaluate_r2(best_vmax, best_km)
    print(f"The best R2 = {base_r2:.4f}")

    result_dict = {
        "carbon_id": primary_carbon,
        "best_vmax": best_vmax,
        "best_km": best_km,
        "base_r2": base_r2,
        "V_grid": V_grid,
        "K_grid": K_grid,
        "R2_matrix": R2_matrix
    }

    if plot_individual:
        plt.figure(figsize=(10, 8))
        cp = plt.contourf(K_grid, V_grid, R2_matrix, levels=30, cmap='RdYlGn')
        cbar = plt.colorbar(cp)
        cbar.set_label('Goodness-of-fit ($R^2$)', fontsize=15, fontweight='bold', labelpad=15)
        
        plt.contour(K_grid, V_grid, R2_matrix, levels=15, colors='k', linewidths=0.5, alpha=0.5)
        plt.plot(best_km, best_vmax, marker='*', color='#FFD700', markersize=25, 
                 markeredgecolor='black', markeredgewidth=1.5, label='Optimal Parameter Set')

        plt.axvline(best_km, color='black', linestyle='--', alpha=0.6, linewidth=1.5)
        plt.axhline(best_vmax, color='black', linestyle='--', alpha=0.6, linewidth=1.5)

        plt.xlabel('$K_m$ (mM)', fontsize=16, fontweight='bold')
        plt.ylabel('$V_{max}$ (mmol/gDW/h)', fontsize=16, fontweight='bold')
        plt.legend(loc='upper right', fontsize=12, framealpha=0.9, edgecolor='black')
        plt.show()

    return result_dict

### B.fragilis

In [ ]:
# Load experimental data of B.fragilis
BF_exp_df = pd.read_csv('/Users/lishijia/Documents/simulations/mono-species cultivation/Bfragilis_OD_long.txt',
                     sep='\t',header=None,names=['Time','CarbonSource','MeanOD','StdOD'])
BF_exp_df['Time'] = pd.to_numeric(BF_exp_df['Time'], errors='coerce')
BF_exp_df['MeanOD'] = pd.to_numeric(BF_exp_df['MeanOD'], errors='coerce')
BF_exp_df['StdOD'] = pd.to_numeric(BF_exp_df['StdOD'], errors='coerce')
BF_exp_df = BF_exp_df.dropna()

# Asumme the coefficient of transforming ODs to biomass is 0.35 for B.fragilis
BF_OD_biomass_factor = 0.35

In [ ]:
# Get the lag time of B.fragilis when using different carbon substrates
BF_GMC_lag = 0.3971741
BF_2FL_lag = 14.4198823
BF_3FL_lag = 15.0219442
BF_3SL_lag = 1.7059391
BF_6SL_lag = 1.7754076
BF_DFL_lag = 18.2920984
BF_LNT_lag = 0.7046790
BF_LNNT_lag = 0.3320590

In [ ]:
# Accodring to experiment results, extract experimental data under different carbon conditions for B.fragilis
BF_exp_data_GMC = Trans_OD_biomass(BF_exp_df,'GMC',BF_OD_biomass_factor)
BF_exp_data_2FL = Trans_OD_biomass(BF_exp_df,'2FL',BF_OD_biomass_factor)
BF_exp_data_3FL = Trans_OD_biomass(BF_exp_df,'3FL',BF_OD_biomass_factor)
BF_exp_data_3SL = Trans_OD_biomass(BF_exp_df,'3SL',BF_OD_biomass_factor)
BF_exp_data_6SL = Trans_OD_biomass(BF_exp_df,'6SL',BF_OD_biomass_factor)
BF_exp_data_DFL = Trans_OD_biomass(BF_exp_df,'DFL',BF_OD_biomass_factor)
BF_exp_data_LNT = Trans_OD_biomass(BF_exp_df,'LNT',BF_OD_biomass_factor)
BF_exp_data_LNNT = Trans_OD_biomass(BF_exp_df,'LNnT',BF_OD_biomass_factor)

In [ ]:
# Read the model of B.fragilis
Bfr_new = cobra.io.read_sbml_model('/Users/lishijia/Downloads/AGORA2_expansion/AGORA2_with_HMO_degradation/sbml/Bacteroides_fragilis_638R.xml')
# Shut down all exchange reactions in GEMs for subsequent medium-specific simulations
Bfr = Bfr_new.copy()
for ex in Bfr.exchanges:
    ex.lower_bound = 0

In [ ]:
products = ['EX_but(e)','EX_ppa(e)','EX_ac(e)','EX_for(e)','EX_lac_D(e)','EX_lac_L(e)','EX_succ(e)']
BF_YCFA_sub_GMC, BF_YCFA_con_GMC = YCFA_medium_def(Bfr,products,BF_exp_data_GMC,Carbon='GMC',C_con=0)
BF_YCFA_sub_2FL, BF_YCFA_con_2FL = YCFA_medium_def(Bfr,products,BF_exp_data_2FL,Carbon='EX_2fuclac(e)',C_con=28.0)
BF_YCFA_sub_3FL, BF_YCFA_con_3FL = YCFA_medium_def(Bfr,products,BF_exp_data_3FL,Carbon='EX_3fuclac(e)',C_con=28.0)
BF_YCFA_sub_3SL, BF_YCFA_con_3SL = YCFA_medium_def(Bfr,products,BF_exp_data_3SL,Carbon='EX_3slac(e)',C_con=21.6)
BF_YCFA_sub_6SL, BF_YCFA_con_6SL = YCFA_medium_def(Bfr,products,BF_exp_data_6SL,Carbon='EX_6slac(e)',C_con=21.6)
BF_YCFA_sub_DFL, BF_YCFA_con_DFL = YCFA_medium_def(Bfr,products,BF_exp_data_DFL,Carbon='EX_dfuclac(e)',C_con=21.5)
BF_YCFA_sub_LNT, BF_YCFA_con_LNT = YCFA_medium_def(Bfr,products,BF_exp_data_LNT,Carbon='EX_lacnttr(e)',C_con=20.0)
BF_YCFA_sub_LNNT, BF_YCFA_con_LNNT = YCFA_medium_def(Bfr,products,BF_exp_data_LNNT,Carbon='EX_lacnnttr(e)',C_con=20.0)

In [ ]:
# Identify the growth-limit substrate when using GMC as the sole carbon source
BF_limits_subs_GMC = limit_subs_dect(Bfr,BF_YCFA_sub_GMC,BF_YCFA_con_GMC)
BF_limits_subs_GMC

In [ ]:
# Identify the growth-limit substrate when using 2FL as the sole carbon source
BF_limits_subs_2FL = limit_subs_dect(Bfr,BF_YCFA_sub_2FL,BF_YCFA_con_2FL)
BF_limits_subs_2FL

In [ ]:
# Identify the growth-limit substrate when using 3FL as the sole carbon source
BF_limits_subs_3FL = limit_subs_dect(Bfr,BF_YCFA_sub_3FL,BF_YCFA_con_3FL)
BF_limits_subs_3FL

In [ ]:
# Identify the growth-limit substrate when using 3SL as the sole carbon source
BF_limits_subs_3SL = limit_subs_dect(Bfr,BF_YCFA_sub_3SL,BF_YCFA_con_3SL)
BF_limits_subs_3SL

In [ ]:
# Identify the growth-limit substrate when using 6SL as the sole carbon source
BF_limits_subs_6SL = limit_subs_dect(Bfr,BF_YCFA_sub_6SL,BF_YCFA_con_6SL)
BF_limits_subs_6SL

In [ ]:
# Identify the growth-limit substrate when using DFL as the sole carbon source
BF_limits_subs_DFL = limit_subs_dect(Bfr,BF_YCFA_sub_DFL,BF_YCFA_con_DFL)
BF_limits_subs_DFL

In [ ]:
# Identify the growth-limit substrate when using LNT as the sole carbon source
BF_limits_subs_LNT = limit_subs_dect(Bfr,BF_YCFA_sub_LNT,BF_YCFA_con_LNT)
BF_limits_subs_LNT

In [ ]:
# Identify the growth-limit substrate when using LNnT as the sole carbon source
BF_limits_subs_LNNT = limit_subs_dect(Bfr,BF_YCFA_sub_LNNT,BF_YCFA_con_LNNT)
BF_limits_subs_LNNT

In [ ]:
BF_GMC_kenitic = plot_parameter_sensitivity_heatmap(Bfr,BF_exp_data_GMC,BF_GMC_lag,['EX_glc_D(e)','EX_malt(e)'],
                                                    BF_YCFA_sub_GMC,BF_YCFA_con_GMC,substrate_limit_val=6.5,
                                                    vmax_range=(0.001,3,30),km_range=(0.001,20,30))

In [ ]:
%%time
BF_GMC_sens = run_local_sensitivity_analysis(Bfr, BF_exp_data_GMC, BF_GMC_lag, 'EX_glc_D(e)', 
                               BF_YCFA_sub_GMC, BF_YCFA_con_GMC, substrate_limit_val=6.5, 
                               best_vmax=1.093, best_km=0.005, plot_individual=True)

In [ ]:
BF_2FL_kenitic = plot_parameter_sensitivity_heatmap(Bfr,BF_exp_data_2FL,BF_2FL_lag,['EX_2fuclac(e)'],
                                                    BF_YCFA_sub_2FL,BF_YCFA_con_2FL,substrate_limit_val=8,
                                                    vmax_range=(0.001,3,30),km_range=(0.001,20,30))

In [ ]:
%%time
BF_2FL_sens = run_local_sensitivity_analysis(Bfr, BF_exp_data_2FL, BF_2FL_lag, 'EX_2fuclac(e)', 
                               BF_YCFA_sub_2FL, BF_YCFA_con_2FL, substrate_limit_val=8, 
                               best_vmax=2.59, best_km=0.691, plot_individual=True)

In [ ]:
BF_3FL_kenitic = plot_parameter_sensitivity_heatmap(Bfr,BF_exp_data_3FL,BF_3FL_lag,['EX_3fuclac(e)'],
                                                    BF_YCFA_sub_3FL,BF_YCFA_con_3FL,substrate_limit_val=8,
                                                    vmax_range=(0.001,3,30),km_range=(0.1,30,30))

In [ ]:
%%time
BF_3FL_sens = run_local_sensitivity_analysis(Bfr, BF_exp_data_3FL, BF_3FL_lag, 'EX_3fuclac(e)', 
                               BF_YCFA_sub_3FL, BF_YCFA_con_3FL, substrate_limit_val=8, 
                               best_vmax=0.123, best_km=4.623, plot_individual=True)

In [ ]:
BF_3SL_kenitic = plot_parameter_sensitivity_heatmap(Bfr,BF_exp_data_3SL,BF_3SL_lag,['EX_3slac(e)'],
                                                    BF_YCFA_sub_3SL,BF_YCFA_con_3SL,substrate_limit_val=8,
                                                    vmax_range=(1,20,30),km_range=(0.01,20,30))

In [ ]:
%%time
BF_3SL_sens = run_local_sensitivity_analysis(Bfr, BF_exp_data_3SL, BF_3SL_lag, 'EX_3slac(e)', 
                               BF_YCFA_sub_3SL, BF_YCFA_con_3SL, substrate_limit_val=8, 
                               best_vmax=6.86, best_km=11.311, plot_individual=True)

In [ ]:
BF_6SL_kenitic = plot_parameter_sensitivity_heatmap(Bfr,BF_exp_data_6SL,BF_6SL_lag,['EX_6slac(e)'],
                                                    BF_YCFA_sub_6SL,BF_YCFA_con_6SL,substrate_limit_val=8,
                                                    vmax_range=(1,20,30),km_range=(0.01,20,30))

In [ ]:
%%time
BF_6SL_sens = run_local_sensitivity_analysis(Bfr, BF_exp_data_6SL, BF_6SL_lag, 'EX_6slac(e)', 
                               BF_YCFA_sub_6SL, BF_YCFA_con_6SL, substrate_limit_val=8, 
                               best_vmax=10.12, best_km=19.311, plot_individual=True)

In [ ]:
BF_DFL_kenitic = plot_parameter_sensitivity_heatmap(Bfr,BF_exp_data_DFL,BF_DFL_lag,['EX_dfuclac(e)'],
                                                    BF_YCFA_sub_DFL,BF_YCFA_con_DFL,substrate_limit_val=8,
                                                    vmax_range=(1,20,30),km_range=(0.01,20,30))

In [ ]:
%%time
BF_DFL_sens = run_local_sensitivity_analysis(Bfr, BF_exp_data_DFL, BF_DFL_lag, 'EX_dfuclac(e)', 
                               BF_YCFA_sub_DFL, BF_YCFA_con_DFL, substrate_limit_val=8, 
                               best_vmax=5.74, best_km=9.324, plot_individual=True)

In [ ]:
BF_LNT_kenitic = plot_parameter_sensitivity_heatmap(Bfr,BF_exp_data_LNT,BF_LNT_lag,['EX_lacnttr(e)'],
                                                    BF_YCFA_sub_LNT,BF_YCFA_con_LNT,substrate_limit_val=8,
                                                    vmax_range=(1,20,30),km_range=(0.01,20,30))

In [ ]:
%%time
BF_LNT_sens = run_local_sensitivity_analysis(Bfr, BF_exp_data_LNT, BF_LNT_lag, 'EX_lacnttr(e)', 
                               BF_YCFA_sub_LNT, BF_YCFA_con_LNT, substrate_limit_val=8, 
                               best_vmax=5.82, best_km=9.311, plot_individual=True)

In [ ]:
BF_LNNT_kenitic = plot_parameter_sensitivity_heatmap(Bfr,BF_exp_data_LNNT,BF_LNNT_lag,['EX_lacnnttr(e)'],
                                                     BF_YCFA_sub_LNNT,BF_YCFA_con_LNNT,substrate_limit_val=8,
                                                     vmax_range=(1,20,30),km_range=(0.01,20,30))

In [ ]:
%%time
BF_LNNT_sens = run_local_sensitivity_analysis(Bfr, BF_exp_data_LNNT, BF_LNNT_lag, 'EX_lacnnttr(e)', 
                               BF_YCFA_sub_LNNT, BF_YCFA_con_LNNT, substrate_limit_val=8, 
                               best_vmax=10.41, best_km=9.85, plot_individual=True)

### B.infantis

In [ ]:
# Load experimental data of B.infantis
BI_exp_df = pd.read_csv('/Users/lishijia/Documents/simulations/mono-species cultivation/Binfantis_OD_long.txt',
                     sep='\t',header=None,names=['Time','CarbonSource','MeanOD','StdOD'])
BI_exp_df['Time'] = pd.to_numeric(BI_exp_df['Time'], errors='coerce')
BI_exp_df['MeanOD'] = pd.to_numeric(BI_exp_df['MeanOD'], errors='coerce')
BI_exp_df['StdOD'] = pd.to_numeric(BI_exp_df['StdOD'], errors='coerce')
BI_exp_df = BI_exp_df.dropna()

# Asumme the coefficient of transforming ODs to biomass is 0.35 for B.infantis
BI_OD_biomass_factor = 0.35

In [ ]:
# Get the lag time of B.infantis when using different carbon substrates
BI_GMC_lag = 0.0
BI_2FL_lag = 8.520838
BI_3FL_lag = 2.148073
BI_DFL_lag = 8.5063248
BI_LNT_lag = 1.044867
BI_LNNT_lag = 2.518846

In [ ]:
# Accodring to experiment results, extract experimental data under different carbon conditions for B.fragilis
BI_exp_data_GMC = Trans_OD_biomass(BI_exp_df,'GMC',BI_OD_biomass_factor)
BI_exp_data_2FL = Trans_OD_biomass(BI_exp_df,'2FL',BI_OD_biomass_factor)
BI_exp_data_3FL = Trans_OD_biomass(BI_exp_df,'3FL',BI_OD_biomass_factor)
BI_exp_data_DFL = Trans_OD_biomass(BI_exp_df,'DFL',BI_OD_biomass_factor)
BI_exp_data_LNT = Trans_OD_biomass(BI_exp_df,'LNT',BI_OD_biomass_factor)
BI_exp_data_LNNT = Trans_OD_biomass(BI_exp_df,'LNnT',BI_OD_biomass_factor)

In [ ]:
# Read the model of B.infantis
Bin_new = cobra.io.read_sbml_model('/Users/lishijia/Downloads/AGORA2_expansion/AGORA2_with_HMO_degradation/sbml/Bifidobacterium_longum_infantis_ATCC_15697.xml')
Bin = Bin_new.copy()
for ex in Bin.exchanges:
    ex.lower_bound = 0.0

In [ ]:
products = ['EX_but(e)','EX_ppa(e)','EX_ac(e)','EX_for(e)','EX_lac_D(e)','EX_lac_L(e)','EX_succ(e)']
BI_YCFA_sub_GMC, BI_YCFA_con_GMC = YCFA_medium_def(Bin,products,BI_exp_data_GMC,Carbon='GMC',C_con=0)
BI_YCFA_sub_2FL, BI_YCFA_con_2FL = YCFA_medium_def(Bin,products,BI_exp_data_2FL,Carbon='EX_2fuclac(e)',C_con=28.0)
BI_YCFA_sub_3FL, BI_YCFA_con_3FL = YCFA_medium_def(Bin,products,BI_exp_data_3FL,Carbon='EX_3fuclac(e)',C_con=28.0)
BI_YCFA_sub_DFL, BI_YCFA_con_DFL = YCFA_medium_def(Bin,products,BI_exp_data_DFL,Carbon='EX_dfuclac(e)',C_con=21.5)
BI_YCFA_sub_LNT, BI_YCFA_con_LNT = YCFA_medium_def(Bin,products,BI_exp_data_LNT,Carbon='EX_lacnttr(e)',C_con=20.0)
BI_YCFA_sub_LNNT, BI_YCFA_con_LNNT = YCFA_medium_def(Bin,products,BI_exp_data_LNNT,Carbon='EX_lacnnttr(e)',C_con=20.0)

In [ ]:
# Identify the growth-limit substrate of B.infantis when using GMC as the sole carbon source
BI_limits_subs_GMC = limit_subs_dect(Bin,BI_YCFA_sub_GMC,BI_YCFA_con_GMC)

# Identify the growth-limit substrate of B.infantis when using 2FL as the sole carbon source
BI_limits_subs_2FL = limit_subs_dect(Bin,BI_YCFA_sub_2FL,BI_YCFA_con_2FL)

# Identify the growth-limit substrate of B.infantis when using 3FL as the sole carbon source
BI_limits_subs_3FL = limit_subs_dect(Bin,BI_YCFA_sub_3FL,BI_YCFA_con_3FL)

# Identify the growth-limit substrate of B.infantis when using DFL as the sole carbon source
BI_limits_subs_DFL = limit_subs_dect(Bin,BI_YCFA_sub_DFL,BI_YCFA_con_DFL)

# Identify the growth-limit substrate of B.infantis when using LNT as the sole carbon source
BI_limits_subs_LNT = limit_subs_dect(Bin,BI_YCFA_sub_LNT,BI_YCFA_con_LNT)

# Identify the growth-limit substrate of B.infantis when using LNNT as the sole carbon source
BI_limits_subs_LNNT = limit_subs_dect(Bin,BI_YCFA_sub_LNNT,BI_YCFA_con_LNNT)

In [ ]:
nh4_idx = BI_YCFA_sub_GMC.index('EX_nh4(e)')
BI_YCFA_con_GMC_rich = BI_YCFA_con_GMC.copy()
BI_YCFA_con_GMC_rich[nh4_idx] = BI_YCFA_con_GMC[nh4_idx]*10
BI_GMC_kenitic = plot_parameter_sensitivity_heatmap(Bin,BI_exp_data_GMC,BI_GMC_lag,['EX_glc_D(e)','EX_malt(e)'],
                                                    BI_YCFA_sub_GMC,BI_YCFA_con_GMC_rich,substrate_limit_val=6.5,
                                                    vmax_range=(1,20,30),km_range=(0.01,20,30))

In [ ]:
%%time
BI_GMC_sens = run_local_sensitivity_analysis(Bin, BI_exp_data_GMC, BI_GMC_lag, 'EX_glc_D(e)', 
                               BI_YCFA_sub_GMC, BI_YCFA_con_GMC_rich, substrate_limit_val=6.5, 
                               best_vmax=2.31, best_km=0.029, plot_individual=True)

In [ ]:
nh4_idx = BI_YCFA_sub_2FL.index('EX_nh4(e)')
BI_YCFA_con_2FL_rich = BI_YCFA_con_2FL.copy()
BI_YCFA_con_2FL_rich[nh4_idx] = BI_YCFA_con_2FL[nh4_idx]*10
BI_2FL_kenitic = plot_parameter_sensitivity_heatmap(Bin,BI_exp_data_2FL,BI_2FL_lag,['EX_2fuclac(e)'],
                                                    BI_YCFA_sub_2FL,BI_YCFA_con_2FL_rich,substrate_limit_val=8,
                                                    vmax_range=(1,20,30),km_range=(0.01,20,30))

In [ ]:
%%time
BI_2FL_sens = run_local_sensitivity_analysis(Bin, BI_exp_data_2FL, BI_2FL_lag, 'EX_2fuclac(e)', 
                               BI_YCFA_sub_2FL, BI_YCFA_con_2FL_rich, substrate_limit_val=8, 
                               best_vmax=14.10, best_km=18.41, plot_individual=True)

In [ ]:
nh4_idx = BI_YCFA_sub_3FL.index('EX_nh4(e)')
BI_YCFA_con_3FL_rich = BI_YCFA_con_3FL.copy()
BI_YCFA_con_3FL_rich[nh4_idx] = BI_YCFA_con_3FL[nh4_idx]*10
BI_3FL_kenitic = plot_parameter_sensitivity_heatmap(Bin,BI_exp_data_3FL,BI_3FL_lag,['EX_3fuclac(e)'],
                                                    BI_YCFA_sub_3FL,BI_YCFA_con_3FL_rich,substrate_limit_val=8,
                                                    vmax_range=(0.001,3,30),km_range=(0.01,20,30))

In [ ]:
%%time
BI_3FL_sens = run_local_sensitivity_analysis(Bin, BI_exp_data_3FL, BI_3FL_lag, 'EX_3fuclac(e)', 
                               BI_YCFA_sub_3FL, BI_YCFA_con_3FL_rich, substrate_limit_val=8, 
                               best_vmax=0.27, best_km=23.62, plot_individual=True)

In [ ]:
nh4_idx = BI_YCFA_sub_DFL.index('EX_nh4(e)')
BI_YCFA_con_DFL_rich = BI_YCFA_con_DFL.copy()
BI_YCFA_con_DFL_rich[nh4_idx] = BI_YCFA_con_DFL[nh4_idx]*10
BI_DFL_kenitic = plot_parameter_sensitivity_heatmap(Bin,BI_exp_data_DFL,BI_DFL_lag,['EX_dfuclac(e)'],
                                                    BI_YCFA_sub_DFL,BI_YCFA_con_DFL_rich,substrate_limit_val=8,
                                                    vmax_range=(0.001,3,30),km_range=(0.01,20,30))

In [ ]:
%%time
BI_DFL_sens = run_local_sensitivity_analysis(Bin, BI_exp_data_DFL, BI_DFL_lag, 'EX_dfuclac(e)', 
                               BI_YCFA_sub_DFL, BI_YCFA_con_DFL_rich, substrate_limit_val=8, 
                               best_vmax=0.11, best_km=12.971, plot_individual=True)

In [ ]:
hxan_idx = BI_YCFA_sub_LNT.index('EX_hxan(e)')
BI_YCFA_con_LNT_rich = BI_YCFA_con_LNT.copy()
BI_YCFA_con_LNT_rich[hxan_idx] = BI_YCFA_con_LNT[hxan_idx]*10
BI_LNT_kenitic = plot_parameter_sensitivity_heatmap(Bin,BI_exp_data_LNT,BI_LNT_lag,['EX_lacnttr(e)'],
                                                    BI_YCFA_sub_LNT,BI_YCFA_con_LNT_rich,substrate_limit_val=8,
                                                    vmax_range=(1,20,30),km_range=(0.01,20,30))

In [ ]:
%%time
BI_LNT_sens = run_local_sensitivity_analysis(Bin, BI_exp_data_LNT, BI_LNT_lag, 'EX_lacnttr(e)', 
                               BI_YCFA_sub_LNT, BI_YCFA_con_LNT_rich, substrate_limit_val=8, 
                               best_vmax=7.83, best_km=15.382, plot_individual=True)

In [ ]:
hxan_idx = BI_YCFA_sub_DFL.index('EX_hxan(e)')
BI_YCFA_con_LNNT_rich = BI_YCFA_con_LNNT.copy()
BI_YCFA_con_LNNT_rich[hxan_idx] = BI_YCFA_con_LNNT[hxan_idx]*10
BI_LNNT_kenitic = plot_parameter_sensitivity_heatmap(Bin,BI_exp_data_LNNT,BI_LNNT_lag,['EX_lacnnttr(e)'],
                                                    BI_YCFA_sub_LNNT,BI_YCFA_con_LNNT_rich,substrate_limit_val=8,
                                                    vmax_range=(1,20,30),km_range=(0.01,20,30))

In [ ]:
%%time
BI_LNNT_sens = run_local_sensitivity_analysis(Bin, BI_exp_data_LNNT, BI_LNNT_lag, 'EX_lacnnttr(e)', 
                               BI_YCFA_sub_LNNT, BI_YCFA_con_LNNT_rich, substrate_limit_val=8, 
                               best_vmax=16.07, best_km=19.77, plot_individual=True)

### B.breve

In [ ]:
# Load experimental data of B.breve
BB_exp_df = pd.read_csv('/Users/lishijia/Documents/simulations/mono-species cultivation/Bbreve_OD_long.txt',
                     sep='\t',header=None,names=['Time','CarbonSource','MeanOD','StdOD'])
BB_exp_df['Time'] = pd.to_numeric(BB_exp_df['Time'], errors='coerce')
BB_exp_df['MeanOD'] = pd.to_numeric(BB_exp_df['MeanOD'], errors='coerce')
BB_exp_df['StdOD'] = pd.to_numeric(BB_exp_df['StdOD'], errors='coerce')
BB_exp_df = BB_exp_df.dropna()

# Asumme the coefficient of transforming ODs to biomass is 0.35 for B.breve
BB_OD_biomass_factor = 0.35

In [ ]:
# Get the lag time of B.breve when using different carbon substrates
BB_GMC_lag = 0.0
BB_LNT_lag = 0.0
BB_LNNT_lag = 38.47537

In [ ]:
# Accodring to experiment results, extract experimental data under different carbon conditions for B.breve
BB_exp_data_GMC = Trans_OD_biomass(BB_exp_df,'GMC',BB_OD_biomass_factor)
BB_exp_data_LNT = Trans_OD_biomass(BB_exp_df,'LNT',BB_OD_biomass_factor)
BB_exp_data_LNNT = Trans_OD_biomass(BB_exp_df,'LNnT',BB_OD_biomass_factor)

In [ ]:
# Read the model of B.breve
Bbr_new = cobra.io.read_sbml_model('/Users/lishijia/Downloads/AGORA2_expansion/AGORA2_with_HMO_degradation/sbml/Bifidobacterium_breve_DSM_20213.xml')
Bbr = Bbr_new.copy()
for ex in Bbr.exchanges:
    ex.lower_bound = 0.0

In [ ]:
products = ['EX_but(e)','EX_ppa(e)','EX_ac(e)','EX_for(e)','EX_lac_D(e)','EX_lac_L(e)','EX_succ(e)']
BB_YCFA_sub_GMC, BB_YCFA_con_GMC = YCFA_medium_def(Bbr,products,BB_exp_data_GMC,Carbon='GMC',C_con=0)
BB_YCFA_sub_LNT, BB_YCFA_con_LNT = YCFA_medium_def(Bbr,products,BB_exp_data_LNT,Carbon='EX_lacnttr(e)',C_con=20.0)
BB_YCFA_sub_LNNT, BB_YCFA_con_LNNT = YCFA_medium_def(Bbr,products,BB_exp_data_LNNT,Carbon='EX_lacnnttr(e)',C_con=20.0)

In [ ]:
# Identify the growth-limit substrate of B.breve when using GMC as the sole carbon source
BB_limits_subs_GMC = limit_subs_dect(Bbr,BB_YCFA_sub_GMC,BB_YCFA_con_GMC)

# Identify the growth-limit substrate of B.breve when using LNT as the sole carbon source
BB_limits_subs_LNT = limit_subs_dect(Bbr,BB_YCFA_sub_LNT,BB_YCFA_con_LNT)

# Identify the growth-limit substrate of B.breve when using LNNT as the sole carbon source
BB_limits_subs_LNNT = limit_subs_dect(Bbr,BB_YCFA_sub_LNNT,BB_YCFA_con_LNNT)

In [ ]:
abz_idx = BB_YCFA_sub_GMC.index('EX_4abz(e)')
BB_YCFA_con_GMC_rich = BB_YCFA_con_GMC.copy()
BB_YCFA_con_GMC_rich[abz_idx] = BB_YCFA_con_GMC[abz_idx]*200
BB_GMC_kenitic = plot_parameter_sensitivity_heatmap(Bbr,BB_exp_data_GMC,BB_GMC_lag,['EX_glc_D(e)','EX_malt(e)'],
                                                    BB_YCFA_sub_GMC,BB_YCFA_con_GMC_rich,substrate_limit_val=6.5,
                                                    vmax_range=(0.1,10,30),km_range=(0.001,5,30))

In [ ]:
%%time
BB_GMC_sens = run_local_sensitivity_analysis(Bbr, BB_exp_data_GMC, BB_GMC_lag, 'EX_glc_D(e)', 
                               BB_YCFA_sub_GMC, BB_YCFA_con_GMC_rich, substrate_limit_val=6.5, 
                               best_vmax=1.81, best_km=0.0034, plot_individual=True)

In [ ]:
abz_idx = BB_YCFA_sub_LNT.index('EX_4abz(e)')
BB_YCFA_con_LNT_rich = BB_YCFA_con_LNT.copy()
#BB_YCFA_con_LNT_rich[abz_idx] = BB_YCFA_con_LNT[abz_idx]*200
BB_YCFA_con_LNT_rich[abz_idx] = 1
BB_LNT_kenitic = plot_parameter_sensitivity_heatmap(Bbr,BB_exp_data_LNT,BB_LNT_lag,['EX_lacnttr(e)'],
                                                    BB_YCFA_sub_LNT,BB_YCFA_con_LNT_rich,substrate_limit_val=8,
                                                    vmax_range=(15,30,30),km_range=(0.01,20,30))

In [ ]:
%%time
BB_LNT_sens = run_local_sensitivity_analysis(Bbr, BB_exp_data_LNT, BB_LNT_lag,['EX_lacnttr(e)'], 
                               BB_YCFA_sub_LNT, BB_YCFA_con_LNT_rich, substrate_limit_val=8, 
                               best_vmax=19.66, best_km=20.00, plot_individual=True)

In [ ]:
abz_idx = BB_YCFA_sub_LNNT.index('EX_4abz(e)')
BB_YCFA_con_LNNT_rich = BB_YCFA_con_LNNT.copy()
#BB_YCFA_con_LNNT_rich[abz_idx] = BB_YCFA_con_LNNT[abz_idx]*200
BB_YCFA_con_LNNT_rich[abz_idx] = 1
BB_LNNT_kenitic = plot_parameter_sensitivity_heatmap(Bbr,BB_exp_data_LNNT,BB_LNNT_lag,['EX_lacnnttr(e)'],
                                                     BB_YCFA_sub_LNNT,BB_YCFA_con_LNNT_rich,substrate_limit_val=20,
                                                     vmax_range=(1,20,30),km_range=(0.01,20,30))

In [ ]:
%%time
BB_LNNT_sens = run_local_sensitivity_analysis(Bbr, BB_exp_data_LNNT, BB_LNNT_lag, ['EX_lacnnttr(e)'], 
                               BB_YCFA_sub_LNNT, BB_YCFA_con_LNNT_rich, substrate_limit_val=20, 
                               best_vmax=0.97, best_km=27.30, plot_individual=True)

### B.pseudocatenulatum

In [ ]:
# Load experimental data of B.pseudo
BP_exp_df = pd.read_csv('/Users/lishijia/Documents/simulations/mono-species cultivation/Bpseudo_OD_long.txt',
                     sep='\t',header=None,names=['Time','CarbonSource','MeanOD','StdOD'])
BP_exp_df['Time'] = pd.to_numeric(BP_exp_df['Time'], errors='coerce')
BP_exp_df['MeanOD'] = pd.to_numeric(BP_exp_df['MeanOD'], errors='coerce')
BP_exp_df['StdOD'] = pd.to_numeric(BP_exp_df['StdOD'], errors='coerce')
BP_exp_df = BP_exp_df.dropna()

# Asumme the coefficient of transforming ODs to biomass is 0.35 for B.breve
BP_OD_biomass_factor = 0.35

In [ ]:
# Get the lag time of B.pseudo when using different carbon substrates
BP_GMC_lag = 0.0
BP_3FL_lag = 31.015620
BP_LNT_lag = 0.7272656

In [ ]:
# Accodring to experiment results, extract experimental data under different carbon conditions for B.pseudo
BP_exp_data_GMC = Trans_OD_biomass(BP_exp_df,'GMC',BP_OD_biomass_factor)
BP_exp_data_3FL = Trans_OD_biomass(BP_exp_df,'3FL',BP_OD_biomass_factor)
BP_exp_data_LNT = Trans_OD_biomass(BP_exp_df,'LNT',BP_OD_biomass_factor)

In [ ]:
Bps_new = cobra.io.read_sbml_model('/Users/lishijia/Downloads/AGORA2_expansion/AGORA2_with_HMO_degradation/sbml/Bifidobacterium_pseudocatenulatum_DSM_20438.xml')
Bps = Bps_new.copy()
for ex in Bps.exchanges:
    ex.lower_bound = 0.0

In [ ]:
products = ['EX_but(e)','EX_ppa(e)','EX_ac(e)','EX_for(e)','EX_lac_D(e)','EX_lac_L(e)','EX_succ(e)']
BP_YCFA_sub_GMC, BP_YCFA_con_GMC = YCFA_medium_def(Bps,products,BP_exp_data_GMC,Carbon='GMC',C_con=0)
BP_YCFA_sub_3FL, BP_YCFA_con_3FL = YCFA_medium_def(Bps,products,BP_exp_data_3FL,Carbon='EX_3fuclac(e)',C_con=20.0)
BP_YCFA_sub_LNT, BP_YCFA_con_LNT = YCFA_medium_def(Bps,products,BP_exp_data_LNT,Carbon='EX_lacnttr(e)',C_con=28.0)

In [ ]:
# Identify the growth-limit substrate of B.pseudo when using GMC as the sole carbon source
BP_limits_subs_GMC = limit_subs_dect(Bps,BP_YCFA_sub_GMC,BP_YCFA_con_GMC)
BP_limits_subs_GMC

# Identify the growth-limit substrate of B.pseudo when using 3FL as the sole carbon source
BP_limits_subs_3FL = limit_subs_dect(Bps,BP_YCFA_sub_3FL,BP_YCFA_con_3FL)
BP_limits_subs_3FL

# Identify the growth-limit substrate of B.pseudo when using LNT as the sole carbon source
BP_limits_subs_LNT = limit_subs_dect(Bps,BP_YCFA_sub_LNT,BP_YCFA_con_LNT)
BP_limits_subs_LNT

In [ ]:
asn_idx = BP_YCFA_sub_GMC.index('EX_asn_L(e)')
asp_idx = BP_YCFA_sub_GMC.index('EX_asp_L(e)')
BP_YCFA_con_GMC_rich = BP_YCFA_con_GMC.copy()
BP_YCFA_con_GMC_rich[asn_idx] = BP_YCFA_con_GMC[asn_idx]*10
BP_YCFA_con_GMC_rich[asp_idx] = BP_YCFA_con_GMC[asp_idx]*10
BP_GMC_kenitic = plot_parameter_sensitivity_heatmap(Bps,BP_exp_data_GMC,BP_GMC_lag,['EX_glc_D(e)','EX_malt(e)'],
                                                    BP_YCFA_sub_GMC,
                                                    BP_YCFA_con_GMC_rich,
                                                    #BP_YCFA_con_GMC,
                                                    substrate_limit_val=6.5,
                                                    vmax_range=(0.001,3,30),km_range=(0.1,10,30))

In [ ]:
%%time
BP_GMC_sens = run_local_sensitivity_analysis(Bps, BP_exp_data_GMC, BP_GMC_lag, 'EX_glc_D(e)', 
                               BP_YCFA_sub_GMC, BP_YCFA_con_GMC_rich, substrate_limit_val=6.5, 
                               best_vmax=0.10, best_km=2.490, plot_individual=True)

In [ ]:
ade_idx = BP_YCFA_sub_3FL.index('EX_ade(e)')
BP_YCFA_con_3FL_rich = BP_YCFA_con_3FL.copy()
BP_YCFA_con_3FL_rich[ade_idx] = BP_YCFA_con_3FL[ade_idx]*10
BP_3FL_kenitic = plot_parameter_sensitivity_heatmap(Bps,BP_exp_data_3FL,BP_3FL_lag,['EX_3fuclac(e)'],
                                                    BP_YCFA_sub_3FL,BP_YCFA_con_3FL_rich,substrate_limit_val=8,
                                                    vmax_range=(0.001,3,30),km_range=(0.001,3,30))

In [ ]:
%%time
BP_3FL_sens = run_local_sensitivity_analysis(Bps, BP_exp_data_3FL, BP_3FL_lag, 'EX_3fuclac(e)', 
                               BP_YCFA_sub_3FL, BP_YCFA_con_3FL_rich, substrate_limit_val=8, 
                               best_vmax=1.04, best_km=0.0008, plot_individual=True)

In [ ]:
ade_idx = BP_YCFA_sub_LNT.index('EX_ade(e)')
thr_idx = BP_YCFA_sub_LNT.index('EX_thr_L(e)')
BP_YCFA_con_LNT_rich = BP_YCFA_con_LNT.copy()
BP_YCFA_con_LNT_rich[ade_idx] = BP_YCFA_con_LNT[ade_idx]*10
BP_YCFA_con_LNT_rich[thr_idx] = BP_YCFA_con_LNT[thr_idx]*10
BP_LNT_kenitic = plot_parameter_sensitivity_heatmap(Bps,BP_exp_data_LNT,BP_LNT_lag,['EX_lacnttr(e)'],
                                                    BP_YCFA_sub_LNT,BP_YCFA_con_LNT_rich,substrate_limit_val=8,
                                                    vmax_range=(1,20,30),km_range=(0.01,20,30))

In [ ]:
%%time
BP_LNT_sens = run_local_sensitivity_analysis(Bps, BP_exp_data_LNT, BP_LNT_lag, 'EX_lacnttr(e)', 
                               BP_YCFA_sub_LNT, BP_YCFA_con_LNT_rich, substrate_limit_val=8, 
                               best_vmax=3.62, best_km=1.389, plot_individual=True)

### B.luti

In [ ]:
# Load experimental data of B.luti
BL_exp_df = pd.read_csv('/Users/lishijia/Documents/simulations/mono-species cultivation/Bluti_OD_long.txt',
                     sep='\t',header=None,names=['Time','CarbonSource','MeanOD','StdOD'])
BL_exp_df['Time'] = pd.to_numeric(BL_exp_df['Time'], errors='coerce')
BL_exp_df['MeanOD'] = pd.to_numeric(BL_exp_df['MeanOD'], errors='coerce')
BL_exp_df['StdOD'] = pd.to_numeric(BL_exp_df['StdOD'], errors='coerce')
BL_exp_df = BL_exp_df.dropna()

# Asumme the coefficient of transforming ODs to biomass is 0.35 for B.luti
BL_OD_biomass_factor = 0.35

In [ ]:
# Get the lag time of B.luti when using different carbon substrates
BL_GMC_lag = 2.7086817

In [ ]:
# Accodring to experiment results, extract experimental data under different carbon conditions for B.pseudo
BL_exp_data_GMC = Trans_OD_biomass(BL_exp_df,'GMC',BL_OD_biomass_factor)

In [ ]:
# read the model of B.luti
Blu_old = cobra.io.read_sbml_model('/Users/lishijia/Documents/simulations/mono-species cultivation/B.luti/Blautia_luti_DSM_14534.xml')
Blu = Blu_old.copy()
for ex in Blu.exchanges:
    ex.lower_bound = 0.0

In [ ]:
products = ['EX_but(e)','EX_ppa(e)','EX_ac(e)','EX_for(e)','EX_lac_D(e)','EX_lac_L(e)','EX_succ(e)']
BL_YCFA_sub_GMC, BL_YCFA_con_GMC = YCFA_medium_def(Blu,products,BL_exp_data_GMC,Carbon='GMC',C_con=0)

In [ ]:
# Identify the growth-limit substrate of B.pseudo when using GMC as the sole carbon source
BL_limits_subs_GMC = limit_subs_dect(Blu,BL_YCFA_sub_GMC,BL_YCFA_con_GMC)

In [ ]:
tyr_idx = BL_YCFA_sub_GMC.index('EX_tyr_L(e)')
BL_YCFA_con_GMC_rich = BL_YCFA_con_GMC.copy()
BL_YCFA_con_GMC_rich[tyr_idx] = 1
BL_limits_subs_GMC = limit_subs_dect(Blu,BL_YCFA_sub_GMC,BL_YCFA_con_GMC_rich)
BL_limits_subs_GMC

In [ ]:
tyr_idx = BL_YCFA_sub_GMC.index('EX_tyr_L(e)')
BL_YCFA_con_GMC_rich = BL_YCFA_con_GMC.copy()
BL_YCFA_con_GMC_rich[tyr_idx] = 1
BL_GMC_kenitic = plot_parameter_sensitivity_heatmap(Blu,BL_exp_data_GMC,BL_GMC_lag,['EX_cellb(e)','EX_glc_D(e)','EX_malt(e)'],
                                                    BL_YCFA_sub_GMC,BL_YCFA_con_GMC_rich,substrate_limit_val=6.5,
                                                    vmax_range=(0.001,3,30),km_range=(0.01,20,30))

In [ ]:
%%time
BL_GMC_sens = run_local_sensitivity_analysis(Blu, BL_exp_data_GMC, BL_GMC_lag, 'EX_glc_D(e)', 
                               BL_YCFA_sub_GMC, BL_YCFA_con_GMC_rich, substrate_limit_val=6.5, 
                               best_vmax=0.72, best_km=0.0032, plot_individual=True)

### B.bifidum

In [ ]:
# Load experimental data of B.luti
Bb_exp_df = pd.read_csv('/Users/lishijia/Documents/simulations/mono-species cultivation/Bbifidum_OD_long.txt',
                     sep='\t',header=None,names=['Time','CarbonSource','MeanOD','StdOD'])
Bb_exp_df['Time'] = pd.to_numeric(Bb_exp_df['Time'], errors='coerce')
Bb_exp_df['MeanOD'] = pd.to_numeric(Bb_exp_df['MeanOD'], errors='coerce')
Bb_exp_df['StdOD'] = pd.to_numeric(Bb_exp_df['StdOD'], errors='coerce')
Bb_exp_df = Bb_exp_df.dropna()

# Asumme the coefficient of transforming ODs to biomass is 0.35 for B.breve
Bb_OD_biomass_factor = 0.35

In [ ]:
# Get the lag time of B.breve when using different carbon substrates
Bb_GMC_lag = 0.0

In [ ]:
# Accodring to experiment results, extract experimental data under different carbon conditions for B.pseudo
Bb_exp_data_GMC = Trans_OD_biomass(Bb_exp_df,'GMC',Bb_OD_biomass_factor)

In [ ]:
# Read the model of B.bifidum
Bbi_new = cobra.io.read_sbml_model('/Users/lishijia/Downloads/AGORA2_expansion/AGORA2_with_HMO_degradation/sbml/Bifidobacterium_bifidum_ATCC_29521.xml')
Bbi = Bbi_new.copy()
for ex in Bbi.exchanges:
    ex.lower_bound = 0

In [ ]:
products = ['EX_but(e)','EX_ppa(e)','EX_ac(e)','EX_for(e)','EX_lac_D(e)','EX_lac_L(e)','EX_succ(e)']
Bb_YCFA_sub_GMC, Bb_YCFA_con_GMC = YCFA_medium_def(Bbi,products,Bb_exp_data_GMC,Carbon='GMC',C_con=0)

In [ ]:
# Identify the growth-limit substrate of B.bifidum when using GMC as the sole carbon source
Bb_limits_subs_GMC = limit_subs_dect(Bbi,Bb_YCFA_sub_GMC,Bb_YCFA_con_GMC)

In [ ]:
tyr_idx = Bb_YCFA_sub_GMC.index('EX_tyr_L(e)')
Bb_YCFA_con_GMC_rich = Bb_YCFA_con_GMC.copy()
Bb_YCFA_con_GMC_rich[tyr_idx] = Bb_YCFA_con_GMC[tyr_idx]*10
Bb_GMC_kenitic = plot_parameter_sensitivity_heatmap(Bbi,Bb_exp_data_GMC,Bb_GMC_lag,['EX_glc_D(e)','EX_malt(e)'],
                                                    Bb_YCFA_sub_GMC,Bb_YCFA_con_GMC_rich,substrate_limit_val=2.5,
                                                    vmax_range=(0.001,3,30),km_range=(0.01,20,30))

In [ ]:
%%time
Bb_GMC_sens = run_local_sensitivity_analysis(Bbi, Bb_exp_data_GMC, Bb_GMC_lag, ['EX_glc_D(e)','EX_malt(e)'], 
                               Bb_YCFA_sub_GMC, Bb_YCFA_con_GMC_rich, substrate_limit_val=2.5, 
                               best_vmax=0.41, best_km=0.399, plot_individual=True)

### A.caccae

In [ ]:
# Load experimental data of A.caccae
AC_exp_df = pd.read_csv('/Users/lishijia/Documents/simulations/mono-species cultivation/Acaccae_OD_long.txt',
                     sep='\t',header=None,names=['Time','CarbonSource','MeanOD','StdOD'])
AC_exp_df['Time'] = pd.to_numeric(AC_exp_df['Time'], errors='coerce')
AC_exp_df['MeanOD'] = pd.to_numeric(AC_exp_df['MeanOD'], errors='coerce')
AC_exp_df['StdOD'] = pd.to_numeric(AC_exp_df['StdOD'], errors='coerce')
AC_exp_df = AC_exp_df.dropna()

# Asumme the coefficient of transforming ODs to biomass is 0.35 for B.breve
AC_OD_biomass_factor = 0.35

In [ ]:
# Get the lag time of A.caccae when using different carbon substrates
AC_GMC_lag = 0.0

In [ ]:
# Accodring to experiment results, extract experimental data under different carbon conditions for A.caccae
AC_exp_data_GMC = Trans_OD_biomass(AC_exp_df,'GMC',AC_OD_biomass_factor)

In [ ]:
AC_exp_data_2FL = Trans_OD_biomass(AC_exp_df,'2FL',AC_OD_biomass_factor)

In [ ]:
# Read the model of A.caccae
Aca_old = cobra.io.read_sbml_model('/Users/lishijia/Documents/simulations/mono-species cultivation/A.caccae/Anaerostipes_caccae_DSM_14662.xml')
Aca = Aca_old.copy()
for ex in Aca.exchanges:
    ex.lower_bound = 0.0

In [ ]:
products = ['EX_but(e)','EX_ppa(e)','EX_ac(e)','EX_for(e)','EX_lac_D(e)','EX_lac_L(e)','EX_succ(e)']
AC_YCFA_sub_GMC, AC_YCFA_con_GMC = YCFA_medium_def(Aca,products,AC_exp_data_GMC,Carbon='GMC',C_con=0)

In [ ]:
# Identify the growth-limit substrate of B.bifidum when using GMC as the sole carbon source
AC_limits_subs_GMC = limit_subs_dect(Aca,AC_YCFA_sub_GMC,AC_YCFA_con_GMC)

In [ ]:
gln_idx = AC_YCFA_sub_GMC.index('EX_gln_L(e)')
nh4_idx = AC_YCFA_sub_GMC.index('EX_nh4(e)')
ser_idx = AC_YCFA_sub_GMC.index('EX_ser_L(e)')
val_idx = AC_YCFA_sub_GMC.index('EX_val_L(e)')
AC_YCFA_con_GMC_rich = AC_YCFA_con_GMC.copy()
AC_YCFA_con_GMC_rich[gln_idx] = AC_YCFA_con_GMC[gln_idx]*10
AC_YCFA_con_GMC_rich[nh4_idx] = AC_YCFA_con_GMC[nh4_idx]*10
AC_YCFA_con_GMC_rich[ser_idx] = AC_YCFA_con_GMC[ser_idx]*10
AC_YCFA_con_GMC_rich[val_idx] = AC_YCFA_con_GMC[val_idx]*10
AC_GMC_kenitic = plot_parameter_sensitivity_heatmap(Aca,AC_exp_data_GMC,AC_GMC_lag,['EX_glc_D(e)','EX_malt(e)'],
                                                    AC_YCFA_sub_GMC,AC_YCFA_con_GMC_rich,substrate_limit_val=6.5,
                                                    vmax_range=(0.001,3,30),km_range=(0.001,3,30))

In [ ]:
%%time
AC_GMC_sens = run_local_sensitivity_analysis(Aca, AC_exp_data_GMC, AC_GMC_lag, 'EX_glc_D(e)', 
                               AC_YCFA_sub_GMC, AC_YCFA_con_GMC_rich, substrate_limit_val=6.5, 
                               best_vmax=1.14, best_km=0.104, plot_individual=True)

### C.aerofaciens

In [ ]:
# Load experimental data of A.caccae
CA_exp_df = pd.read_csv('/Users/lishijia/Documents/simulations/mono-species cultivation/Caerofa_OD_long.txt',
                     sep='\t',header=None,names=['Time','CarbonSource','MeanOD','StdOD'])
CA_exp_df['Time'] = pd.to_numeric(CA_exp_df['Time'], errors='coerce')
CA_exp_df['MeanOD'] = pd.to_numeric(CA_exp_df['MeanOD'], errors='coerce')
CA_exp_df['StdOD'] = pd.to_numeric(CA_exp_df['StdOD'], errors='coerce')
CA_exp_df = CA_exp_df.dropna()

# Asumme the coefficient of transforming ODs to biomass is 0.35 for B.breve
CA_OD_biomass_factor = 0.35

In [ ]:
# Get the lag time of C.aerofa when using different carbon substrates
CA_GMC_lag = 12.961554

In [ ]:
# Accodring to experiment results, extract experimental data under different carbon conditions for C.aerofa
CA_exp_data_GMC = Trans_OD_biomass(CA_exp_df,'GMC',CA_OD_biomass_factor)

In [ ]:
# Read the model of C.aerofa
Cae_old = cobra.io.read_sbml_model("/Users/lishijia/Documents/simulations/mono-species cultivation/C.aerofaciens/Collinsella_aerofaciens_ATCC_25986.xml")
Cae = Cae_old.copy()
for ex in Cae.exchanges:
    ex.lower_bound = 0

In [ ]:
products = ['EX_but(e)','EX_ppa(e)','EX_ac(e)','EX_for(e)','EX_lac_D(e)','EX_lac_L(e)','EX_succ(e)']
CA_YCFA_sub_GMC, CA_YCFA_con_GMC = YCFA_medium_def(Cae,products,CA_exp_data_GMC,Carbon='GMC',C_con=0)

In [ ]:
# Identify the growth-limit substrate of C.aerofa when using GMC as the sole carbon source
CA_limits_subs_GMC = limit_subs_dect(Cae,CA_YCFA_sub_GMC,CA_YCFA_con_GMC)

In [ ]:
gln_idx = CA_YCFA_sub_GMC.index('EX_gln_L(e)')
val_idx = CA_YCFA_sub_GMC.index('EX_val_L(e)')
CA_YCFA_con_GMC_rich = CA_YCFA_con_GMC.copy()
CA_YCFA_con_GMC_rich[gln_idx] = CA_YCFA_con_GMC[gln_idx]*10
CA_YCFA_con_GMC_rich[val_idx] = CA_YCFA_con_GMC[val_idx]*10
CA_GMC_kenitic = plot_parameter_sensitivity_heatmap(Cae,CA_exp_data_GMC,CA_GMC_lag,['EX_glc_D(e)','EX_malt(e)'],
                                                    CA_YCFA_sub_GMC,CA_YCFA_con_GMC_rich,substrate_limit_val=10.0,
                                                    vmax_range=(0.001,5,30),km_range=(0.001,6,30))

In [ ]:
%%time
CA_GMC_sens = run_local_sensitivity_analysis(Cae, CA_exp_data_GMC, CA_GMC_lag, 'EX_glc_D(e)', 
                               CA_YCFA_sub_GMC, CA_YCFA_con_GMC_rich, substrate_limit_val=10.0, 
                               best_vmax=0.86, best_km=0.828, plot_individual=True)

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['svg.fonttype'] = 'none'

def generate_labels(n):

    labels = []
    for i in range(n):
        if i < 26:
            labels.append(string.ascii_uppercase[i])
        else:
            labels.append(string.ascii_uppercase[(i // 26) - 1] + string.ascii_uppercase[i % 26])
    return labels

def plot_all_strains_heatmaps(all_strains_dict, ncols=4, output_name="All_Strains_Heatmaps"):

    total_plots = sum(len(carbon_dict) for carbon_dict in all_strains_dict.values())
    
    if total_plots == 0:
        print("Error: The input data is NULL")
        return

    nrows = int(np.ceil(total_plots / ncols))
    
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, 
                             figsize=(ncols * 5.5, nrows * 4.5), 
                             constrained_layout=True)

    if total_plots == 1:
        axes = np.array([axes])
    else:
        axes = axes.flatten()
        
    labels = generate_labels(total_plots)
    
    plot_idx = 0
    for strain_name, carbon_dict in all_strains_dict.items():
        for carbon_name, df in carbon_dict.items():
            ax = axes[plot_idx]
            
            vmin = max(0, df.values.min())
            
            sns.heatmap(df, 
                        cmap='RdYlGn', 
                        vmin=vmin, 
                        vmax=1.0, 
                        ax=ax,
                        linewidths=0.5, 
                        linecolor='white',
                        cbar_kws={'label': r'$R^2$'})
            
            ax.invert_yaxis()

            escaped_name = strain_name.replace(" ", r"\ ")
            
            strain_italic = f"$\\mathit{{{escaped_name}}}$"
            
            ax.set_title(f"{strain_italic} ({carbon_name})", fontsize=16, fontweight='bold', pad=12)
            ax.set_xlabel(r'$V_{max}$ (mmol/gDW/h)', fontsize=14, fontweight='bold')
            ax.set_ylabel(r'$K_m$ (mmol/L)', fontsize=14, fontweight='bold')
            
            ax.tick_params(axis='both', which='major', labelsize=12)
            plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')
            plt.setp(ax.get_yticklabels(), rotation=0)

            ax.text(-0.15, 1.05, labels[plot_idx], transform=ax.transAxes, 
                    fontsize=18, fontweight='bold', va='bottom', ha='right')
            
            plot_idx += 1

    for j in range(plot_idx, len(axes)):
        axes[j].set_visible(False)

    # 5. 保存图片
    plt.savefig(f"{output_name}.png", format='png', dpi=600, bbox_inches='tight')
    plt.savefig(f"{output_name}.pdf", format='pdf', bbox_inches='tight')
    
    print(f"Drawing completed！Including {total_plots} heatmaps，saved as {output_name}.png / .pdf")
    plt.show()

In [ ]:
if __name__ == "__main__":
    
    master_data_dict = {
        'B. fragilis': {
            'GMC': BF_GMC_kenitic['heatmap_data'],
            '2FL': BF_2FL_kenitic['heatmap_data'],
            '3FL': BF_3FL_kenitic['heatmap_data'],
            '3SL': BF_3SL_kenitic['heatmap_data'],
            '6SL': BF_6SL_kenitic['heatmap_data'],
            'DFL': BF_DFL_kenitic['heatmap_data'],
            'LNT': BF_LNT_kenitic['heatmap_data'],
            'LNnT': BF_LNNT_kenitic['heatmap_data']
        },
        
        'B. infantis': {
            'GMC': BI_GMC_kenitic['heatmap_data'],
            '2FL': BI_2FL_kenitic['heatmap_data'],
            '3FL': BI_3FL_kenitic['heatmap_data'],
            'DFL': BI_DFL_kenitic['heatmap_data'],
            'LNT': BI_LNT_kenitic['heatmap_data'],
            'LNnT': BI_LNNT_kenitic['heatmap_data']
        },
        
        'B. breve': {
            'GMC': BB_GMC_kenitic['heatmap_data'],
            'LNT': BB_LNT_kenitic['heatmap_data'],
            'LNnT': BB_LNNT_kenitic['heatmap_data']
        },
        
        'B. pseudocatenulatum': {
            'GMC': BP_GMC_kenitic['heatmap_data'],
            'LNT': BP_LNT_kenitic['heatmap_data'],
            '3FL': BP_3FL_kenitic['heatmap_data']
        },

        'B. luti': {
            'GMC': BL_GMC_kenitic['heatmap_data']
        },

        'B. bifidum': {
            'GMC': Bb_GMC_kenitic['heatmap_data']
        },

        'C. aerofaciens': {
            'GMC': CA_GMC_kenitic['heatmap_data']
        },

        'A. caccae': {
            'GMC': AC_GMC_kenitic['heatmap_data']
        }
    }
    
    plot_all_strains_heatmaps(master_data_dict, ncols=4, output_name="All_8_Strains_Heatmaps2")

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['svg.fonttype'] = 'none'

hmo_colors = {
    "2FL"  : "#D55E00", 
    "3FL"  : "#0072B2", 
    "3SL"  : "#E69F00", 
    "6SL"  : "#009E73", 
    "DFL"  : "#CC79A7", 
    "LNT"  : "#56B4E9", 
    "LNnT" : "#CC7",
    "GMC"  : "#000000"
}

def generate_labels(n):
    labels = []
    for i in range(n):
        if i < 26:
            labels.append(string.ascii_uppercase[i])
        else:
            labels.append(string.ascii_uppercase[(i // 26) - 1] + string.ascii_uppercase[i % 26])
    return labels

def plot_all_strains_growth_curves(all_strains_full_dict, ncols=4, output_name="All_Strains_Growth_Curves"):
    total_plots = sum(len(carbon_dict) for carbon_dict in all_strains_full_dict.values())
    if total_plots == 0:
        print("Error: The input data is NULL")
        return

    nrows = int(np.ceil(total_plots / ncols))
    
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, 
                             figsize=(ncols * 6.5, nrows * 4.5), 
                             constrained_layout=True)
    
    if total_plots == 1:
        axes = np.array([axes])
    else:
        axes = axes.flatten()
        
    labels = generate_labels(total_plots)
    
    plot_idx = 0
    for strain_name, carbon_dict in all_strains_full_dict.items():
        for carbon_name, result_dict in carbon_dict.items():
            ax1 = axes[plot_idx]

            df = result_dict['time_series_data']
            
            meta = result_dict.get('metadata', {})
            r2_val = meta.get('max_r2', meta.get('final_r2', meta.get('r2', np.nan)))
            
            current_color = hmo_colors.get(carbon_name, "#555555")
            
            time_col = 'Time'
            exp_col = 'Exp_OD' if 'Exp_OD' in df.columns else 'Exp_Biomass_Norm'
            std_col = 'Exp_OD_Std' if 'Exp_OD_Std' in df.columns else 'Exp_Std_Norm'
            sim_col = 'Sim_Biomass' if 'Sim_Biomass' in df.columns else 'Sim_Biomass_Norm'
            
            t = df[time_col].values
            exp_vals = df[exp_col].values
            std_vals = df[std_col].values
            sim_vals = df[sim_col].values
            
            lns1 = ax1.plot(t, exp_vals, 'o', color=current_color, markersize=5.5, alpha=0.9, label='In Vitro (OD)')
            ax1.fill_between(t, exp_vals - std_vals, exp_vals + std_vals, color=current_color, alpha=0.15)

            ax1.set_xlabel('Time (h)', fontsize=21, fontweight='bold')
            ax1.set_ylabel('Measured $OD_{600}$', color='black', fontsize=21, fontweight='bold')
            ax1.tick_params(axis='y', labelcolor='black', labelsize=19)
            ax1.tick_params(axis='x', labelsize=19)
            
            y1_max = max(exp_vals + std_vals)
            ax1.set_ylim(-0.05 * y1_max, y1_max * 1.2)

            ax2 = ax1.twinx()
            if not np.isnan(r2_val):
                sim_label = f'In Silico ($R^2$={r2_val:.3f})'
            else:
                sim_label = 'In Silico'
                
            lns2 = ax2.plot(t, sim_vals, '-', color=current_color, linewidth=2.5, label=sim_label)
            
            ax2.set_ylabel('Simulated Biomass', color='black', fontsize=21, fontweight='bold')
            ax2.tick_params(axis='y', labelcolor='black', labelsize=19)
            
            y2_max = max(sim_vals) if max(sim_vals) > 0 else 1.0
            ax2.set_ylim(-0.05 * y2_max, y2_max * 1.2)
            
            lns = lns1 + lns2
            labs = [l.get_label() for l in lns]
            ax1.legend(lns, labs, loc='upper left', fontsize=17, frameon=False)
            
            escaped_name = strain_name.replace(" ", r"\ ")
            strain_italic = f"$\\mathit{{{escaped_name}}}$"
            ax1.set_title(f"{strain_italic} ({carbon_name})", fontsize=23, fontweight='bold', pad=12)
            
            ax1.text(-0.15, 1.05, labels[plot_idx], transform=ax1.transAxes, 
                     fontsize=26, fontweight='bold', va='bottom', ha='right')
            
            plot_idx += 1

    for j in range(plot_idx, len(axes)):
        axes[j].set_visible(False)

    plt.savefig(f"{output_name}.png", format='png', dpi=600, bbox_inches='tight')
    plt.savefig(f"{output_name}.pdf", format='pdf', bbox_inches='tight')
    
    print(f"Drawing complete！Including {total_plots} growth curves，Saved as {output_name}.png / .pdf")
    plt.show()

In [ ]:
if __name__ == "__main__":
    
    master_ts_dict = {
        'B. fragilis': {
            'GMC': BF_GMC_kenitic,
            '2FL': BF_2FL_kenitic,
            '3FL': BF_3FL_kenitic,
            '3SL': BF_3SL_kenitic,
            '6SL': BF_6SL_kenitic,
            'DFL': BF_DFL_kenitic,
            'LNT': BF_LNT_kenitic,
            'LNnT': BF_LNNT_kenitic
        },
        
        'B. infantis': {
            'GMC': BI_GMC_kenitic,
            '2FL': BI_2FL_kenitic,
            '3FL': BI_3FL_kenitic,
            'DFL': BI_DFL_kenitic,
            'LNT': BI_LNT_kenitic,
            'LNnT': BI_LNNT_kenitic
        },
        
        'B. breve': {
            'GMC': BB_GMC_kenitic,
            'LNT': BB_LNT_kenitic,
            'LNnT': BB_LNNT_kenitic
        },
        
        'B. pseudocatenulatum': {
            'GMC': BP_GMC_kenitic,
            'LNT': BP_LNT_kenitic,
            '3FL': BP_3FL_kenitic
        },

        'B. luti': {
            'GMC': BL_GMC_kenitic
        },

        'B. bifidum': {
            'GMC': Bb_GMC_kenitic
        },

        'C. aerofaciens': {
            'GMC': CA_GMC_kenitic
        },

        'A. caccae': {
            'GMC': AC_GMC_kenitic
        }
    }
    
    plot_all_strains_growth_curves(master_ts_dict, ncols=4, output_name="All_Strains_Growth_Curves_new")

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['svg.fonttype'] = 'none'

def extract_simulated_endpoints(sim_results_dict, scfa_names):

    carbon_sources = list(sim_results_dict.keys())
    df_sim = pd.DataFrame(index=scfa_names, columns=carbon_sources)
    
    for carbon, res in sim_results_dict.items():
        ts_df = res['time_series_data']
        last_row = ts_df.iloc[-1]
        
        for scfa in scfa_names:
            col_name = f'Sim_{scfa}'
            if col_name in ts_df.columns:
                df_sim.loc[scfa, carbon] = last_row[col_name]
            else:
                df_sim.loc[scfa, carbon] = 0.0
                
    return df_sim.astype(float)

def evaluate_and_plot_qualitative_scfa(df_exp, df_sim, exp_thresh=0.5, sim_thresh=0.01, strain_name="B. fragilis", ax=None):

    common_cols = [c for c in df_exp.columns if c in df_sim.columns]
    common_rows = [r for r in df_exp.index if r in df_sim.index]
    
    if not common_cols or not common_rows:
        raise ValueError("Unmatched experimental data col/rownames of experimental and measured data!")
        
    df_exp_aligned = df_exp.loc[common_rows, common_cols]
    df_sim_aligned = df_sim.loc[common_rows, common_cols]
    
    df_exp_bin = (df_exp_aligned > exp_thresh).astype(int)
    df_sim_bin = (df_sim_aligned > sim_thresh).astype(int)

    y_exp = df_exp_bin.values.flatten()
    y_sim = df_sim_bin.values.flatten()
    tn, fp, fn, tp = confusion_matrix(y_exp, y_sim, labels=[0, 1]).ravel()
    
    accuracy = (tp + tn) / len(y_exp) if len(y_exp) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    mcc = matthews_corrcoef(y_exp, y_sim)
    
    print(f"[{strain_name}] ACC: {accuracy:.1%} | SPEC: {specificity:.1%} | SENS: {sensitivity:.1%} | MCC: {mcc:.3f}")

    df_combined = df_exp_bin * 2 + df_sim_bin
    
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5), dpi=300)
        
    cmap_4state = ListedColormap(["#f0f0f0", "#e41a1c", "#ff7f00", "#2171b5"])
    
    sns.heatmap(df_combined, ax=ax, cmap=cmap_4state, vmin=0, vmax=3, cbar=False, 
                linewidths=1.5, linecolor='white')
    
    ax.set_title(f"*{strain_name}*\nAcc: {accuracy:.1%} | Sens: {sensitivity:.1%}", 
                 fontsize=15, fontweight='bold', pad=12)
    
    ax.set_ylabel("") 
    ax.tick_params(axis='both', which='major', labelsize=12)
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right', rotation_mode='anchor')
    plt.setp(ax.get_yticklabels(), rotation=0)

    return {"Accuracy": accuracy, "Specificity": specificity, "Sensitivity": sensitivity, "MCC": mcc, "TN": tn, "FP": fp, "FN": fn, "TP": tp}

In [ ]:
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['svg.fonttype'] = 'none'

scfa_list = ['Formate', 'Acetate', 'Propionate', 'Butyrate', 'Lactate', 'Succinate']
base_path = "/Users/lishijia/Documents/simulations/Butyrate project/"

strains_config = [
    {
        'file': 'Bfr_hplc.csv',
        'sims': {'GMC': BF_GMC_kenitic, '2FL': BF_2FL_kenitic, '3FL': BF_3FL_kenitic, 'DFL': BF_DFL_kenitic, '3SL': BF_3SL_kenitic, '6SL': BF_6SL_kenitic, 'LNT': BF_LNT_kenitic, 'LNNT': BF_LNNT_kenitic},
        'exp_t': 0.5, 'sim_t': 0.01, 'name': 'B. fragilis'
    },
    {
        'file': 'Bin_hplc.csv',
        'sims': {'GMC': BI_GMC_kenitic, '2FL': BI_2FL_kenitic, '3FL': BI_3FL_kenitic, 'DFL': BI_DFL_kenitic, 'LNT': BI_LNT_kenitic, 'LNNT': BI_LNNT_kenitic},
        'exp_t': 0.1, 'sim_t': 0.001, 'name': 'B. infantis'
    },
    {
        'file': 'Bbr_hplc.csv',
        'sims': {'GMC': BB_GMC_kenitic, 'LNT': BB_LNT_kenitic, 'LNNT': BF_LNNT_kenitic},
        'exp_t': 0.5, 'sim_t': 0.001, 'name': 'B. breve'
    },
    {
        'file': 'Bps_hplc.csv',
        'sims': {'GMC': BP_GMC_kenitic, '3FL': BP_3FL_kenitic, 'LNT': BP_LNT_kenitic},
        'exp_t': 0.1, 'sim_t': 0.001, 'name': 'B. pseudocatenulatum'
    },
    {
        'file': 'Blu_hplc.csv',
        'sims': {'GMC': BL_GMC_kenitic},
        'exp_t': 0.1, 'sim_t': 0.001, 'name': 'B. luti'
    },
    {
        'file': 'Bbi_hplc.csv',
        'sims': {'GMC': Bb_GMC_kenitic},
        'exp_t': 0.1, 'sim_t': 0.001, 'name': 'B. bifidum'
    },
    {
        'file': 'Cae_hplc.csv',
        'sims': {'GMC': CA_GMC_kenitic},
        'exp_t': 0.01, 'sim_t': 0.000001, 'name': 'C. aerofaciens' 
    },
    {
        'file': 'Aca_hplc.csv',
        'sims': {'GMC': AC_GMC_kenitic},
        'exp_t': 0.01, 'sim_t': 0.000001, 'name': 'A. caccae'
    }
]

fig, axes = plt.subplots(2, 4, figsize=(24, 11), dpi=800)
axes = axes.flatten()
panel_labels = string.ascii_uppercase 

all_metrics = {}

for i, config in enumerate(strains_config):
    ax = axes[i]

    file_path = base_path + config['file']
    df_in_vitro = pd.read_csv(file_path, index_col=0)

    df_in_silico = extract_simulated_endpoints(config['sims'], scfa_names=scfa_list)

    metrics = evaluate_and_plot_qualitative_scfa(
        df_exp = df_in_vitro, 
        df_sim = df_in_silico, 
        exp_thresh = config['exp_t'], 
        sim_thresh = config['sim_t'], 
        strain_name = config['name'],
        ax = ax  
    )
    all_metrics[config['name']] = metrics

    if ax.xaxis.get_label().get_text():
        ax.xaxis.label.set_fontsize(ax.xaxis.label.get_fontsize() + 6)
    if ax.yaxis.get_label().get_text():
        ax.yaxis.label.set_fontsize(ax.yaxis.label.get_fontsize() + 6)

    for label in ax.get_xticklabels():
        label.set_fontsize(label.get_fontsize() + 6)
    for label in ax.get_yticklabels():
        label.set_fontsize(label.get_fontsize() + 6)

    ax.text(-0.05, 1.08, panel_labels[i], transform=ax.transAxes, 
            fontsize=24, fontweight='bold', va='bottom', ha='right')
    
tn_patch = mpatches.Patch(color='#f0f0f0', label='True Negative (Both Non-producer)')
tp_patch = mpatches.Patch(color='#2171b5', label='True Positive (Both Producer)')
fp_patch = mpatches.Patch(color='#e41a1c', label='False Positive (Sim Predicted, Exp Not)')
fn_patch = mpatches.Patch(color='#ff7f00', label='False Negative (Exp Produced, Sim Not)')

plt.tight_layout(rect=[0, 0.08, 1, 1]) 
plt.subplots_adjust(hspace=0.35, wspace=0.3)

fig.legend(handles=[tp_patch, tn_patch, fn_patch, fp_patch], 
           loc='lower center', ncol=4, bbox_to_anchor=(0.5, 0.01), 
           fontsize=20, frameon=False)

fig.savefig("/Users/lishijia/Documents/simulations/Butyrate project/Supplementary_Figure2_new.jpeg", bbox_inches="tight")
plt.show()

In [ ]:
def plot_all_strains_sensitivity_contours(master_sens_dict, ncols=4, output_name="All_Strains_Sensitivity_Contours"):
    
    total_plots = sum(len(carbon_dict) for carbon_dict in master_sens_dict.values())
    if total_plots == 0:
        print("错误：输入数据为空！")
        return

    nrows = int(np.ceil(total_plots / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, 
                             figsize=(ncols * 6, nrows * 5), 
                             constrained_layout=True)
    
    if total_plots == 1:
        axes = np.array([axes])
    else:
        axes = axes.flatten()
        
    labels = list(string.ascii_uppercase) * (total_plots // 26 + 1)
    
    plot_idx = 0
    for strain_name, carbon_dict in master_sens_dict.items():
        for carbon_name, result_dict in carbon_dict.items():
            ax = axes[plot_idx]
            
            V_grid = result_dict['V_grid']
            K_grid = result_dict['K_grid']
            R2_matrix = result_dict['R2_matrix']
            best_vmax = result_dict['best_vmax']
            best_km = result_dict['best_km']
            base_r2 = result_dict['base_r2']
            
            cp = ax.contourf(K_grid, V_grid, R2_matrix, levels=30, cmap='RdYlGn')
            ax.contour(K_grid, V_grid, R2_matrix, levels=15, colors='k', linewidths=0.5, alpha=0.5)
            
            ax.plot(best_km, best_vmax, marker='*', color='#FFD700', markersize=20, 
                    markeredgecolor='black', markeredgewidth=1.0)
            ax.axvline(best_km, color='black', linestyle='--', alpha=0.6, linewidth=1.5)
            ax.axhline(best_vmax, color='black', linestyle='--', alpha=0.6, linewidth=1.5)
            
            annotation_text = f"$V_{{max}}$: {best_vmax:.2f}\n$K_m$: {best_km:.3f}"
            ax.annotate(annotation_text, 
                        xy=(best_km, best_vmax), 
                        xytext=(10, -10), 
                        textcoords='offset points',
                        fontsize=14,
                        fontweight='bold',
                        color='black',
                        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.85))
            
            escaped_name = strain_name.replace(" ", r"\ ")
            strain_italic = f"$\\mathit{{{escaped_name}}}$"
            ax.set_title(f"{strain_italic} ({carbon_name})\n$R^2$={base_r2:.3f}", 
                         fontsize=18, fontweight='bold', pad=10)
            
            ax.set_xlabel('$K_m$ (mM)', fontsize=17, fontweight='bold')
            ax.set_ylabel('$V_{max}$ (mmol/gDW/h)', fontsize=17, fontweight='bold')

            ax.tick_params(labelsize=15)

            ax.text(-0.15, 1.05, labels[plot_idx], transform=ax.transAxes, 
                    fontsize=22, fontweight='bold', va='bottom', ha='right')

            cbar = fig.colorbar(cp, ax=ax, fraction=0.046, pad=0.04)
            cbar.ax.tick_params(labelsize=13)
            
            if plot_idx % ncols == ncols - 1 or plot_idx == total_plots - 1:
                 cbar.set_label('$R^2$', fontsize=15, fontweight='bold')
            
            plot_idx += 1

    for j in range(plot_idx, len(axes)):
        axes[j].set_visible(False)

    plt.savefig(f"{output_name}.png", format='png', dpi=600, bbox_inches='tight')
    plt.savefig(f"{output_name}.pdf", format='pdf', bbox_inches='tight')
    plt.show()

In [ ]:
if __name__ == "__main__":
    
    master_sens_dict = {
        'B. fragilis': {
            'GMC': BF_GMC_sens,
            '2FL': BF_2FL_sens,
            '3FL': BF_3FL_sens,
            '3SL': BF_3SL_sens,
            '6SL': BF_6SL_sens,
            'DFL': BF_DFL_sens,
            'LNT': BF_LNT_sens,
            'LNnT': BF_LNNT_sens
        },
        
        'B. infantis': {
            'GMC': BI_GMC_sens,
            '2FL': BI_2FL_sens,
            '3FL': BI_3FL_sens,
            'DFL': BI_DFL_sens,
            'LNT': BI_LNT_sens,
            'LNnT': BI_LNNT_sens
        },
        
        'B. breve': {
            'GMC': BB_GMC_sens,
            'LNT': BB_LNT_sens,
            'LNnT': BB_LNNT_sens
        },
        
        'B. pseudocatenulatum': {
            'GMC': BP_GMC_sens,
            'LNT': BP_LNT_sens,
            '3FL': BP_3FL_sens
        },

        'B. luti': {
            'GMC': BL_GMC_sens
        },

        'B. bifidum': {
            'GMC': Bb_GMC_sens
        },

        'C. aerofaciens': {
            'GMC': CA_GMC_sens
        },

        'A. caccae': {
            'GMC': AC_GMC_sens
        }
    }
    
    plot_all_strains_sensitivity_contours(
    master_sens_dict=master_sens_dict, 
    ncols=4,
    output_name="Supplementary_Figure_Sensitivity")